# Codex Configuration

This notebook inspects the current environment (network reachability, local LLM
services, MCP servers) and reconciles it against `~/.codex/config.toml` and
`.codex/config.toml` in the workspace.

It will:

1. Probe the environment — host gateway, LM Studio (direct + via the namespace-tools shim), Ollama, MCP servers, the `codex` and `codex-acp` binaries.
2. Load and pretty-print the current Codex configs (user-scope + project-scope) plus the active model catalog.
3. Diagnose missing or conflicting entries.
4. *On request* patch the configs in place using `tomlkit`, so comments and any benign settings you've added by hand survive untouched.

The intended default profile after this notebook is one that is known to work
in the **current** environment state — for this container that means
LM Studio + the Responses namespace-tools shim + `qwen/qwen3-coder-next`.

Background reading:
[docs/codex-lmstudio-shim.md](docs/codex-lmstudio-shim.md),
[docs/mcp-config-and-slash-commands.md](docs/mcp-config-and-slash-commands.md).

## 0. Prerequisites

We rely on `tomlkit` for round-trip-preserving TOML edits. It is already pulled
in by Codex / the kernel, but install on demand if missing.

In [ ]:
import importlib, subprocess, sys
for pkg in ("tomlkit",):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

# All logic lives in scripts/codex_configure.py so it can be re-used outside
# the notebook (CI, a shell hook, etc.). The notebook is just a UI for it.
import sys
from pathlib import Path
SCRIPT_DIR = Path("/workspaces/agent-client-kernel/scripts")
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import importlib, codex_configure
importlib.reload(codex_configure)
from codex_configure import (
    USER_CONFIG, PROJECT_CONFIG, CATALOG_FILE,
    LMSTUDIO_DIRECT, LMSTUDIO_VIA_SHIM, SHIM_OFF,
    PREFERRED_MODEL, PREFERRED_PROFILE, PREFERRED_PROVIDER,
    probe, load_configs, diagnose, apply_fixes,
)

print("User config :", USER_CONFIG, "(exists)" if USER_CONFIG.exists() else "(MISSING)")
print("Project cfg :", PROJECT_CONFIG, "(exists)" if PROJECT_CONFIG.exists() else "(missing)")
print("Model catalog:", CATALOG_FILE, "(exists)" if CATALOG_FILE.exists() else "(MISSING)")
print("LM Studio upstream :", LMSTUDIO_DIRECT)
print("LM Studio via shim :", LMSTUDIO_VIA_SHIM, "(shim disabled)" if SHIM_OFF else "")


User config : /home/jovyan/.codex/config.toml (exists)
Project cfg : /workspaces/agent-client-kernel/.codex/config.toml (exists)
Model catalog: /home/jovyan/.codex/model-catalogs/local-lmstudio.json (exists)
LM Studio upstream : http://host.docker.internal:1234
LM Studio via shim : http://127.0.0.1:18234/v1 


## 1. Probe the environment

In [ ]:
import json
env_facts = probe()
print(json.dumps(env_facts, indent=2, default=str))


{
  "host_gateway": "host.docker.internal",
  "host_gateway_resolves": true,
  "lmstudio_direct_url": "http://host.docker.internal:1234",
  "lmstudio_direct_reachable": true,
  "lmstudio_models": [
    "google/gemma-4-31b",
    "unsloth/gemma-4-31b-it",
    "gemma-4-31b",
    "f2llm-v2",
    "qwen3.5-122b-a10b",
    "qwen3.5-27b",
    "qwen3.5-35b-a3b",
    "qwen/qwen3-coder-next",
    "qwen/qwen3-next-80b",
    "llama-3.2-1b-instruct",
    "qwen2.5-0.5b-instruct-mlx",
    "glm-4.7-reap-218b-a32b",
    "glm-4.7-flash",
    "zai-org/glm-4.7-flash",
    "nomic-ai/text-embedding-nomic-embed-text-v1.5",
    "devstral-2-123b-instruct-2512",
    "minimax/minimax-m2",
    "s3dev-ai/text-embedding-nomic-embed-text-v1.5",
    "nomicai-modernbert-embed-base",
    "nvidia/nemotron-3-nano",
    "mistralai/devstral-small-2-2512",
    "deepswe-preview-mlx",
    "devstral-small-2507-mlx",
    "openai/gpt-oss-120b",
    "google/gemma-3n-e4b",
    "qwen/qwen3-coder-30b",
    "gpt-oss-20b-mlx",
    "goo

## 2. Load Codex configuration files

We parse with `tomlkit` so the round-trip preserves comments and any extra keys
you may have added.

In [ ]:
cfg = load_configs()

def _summary(doc, label):
    if doc is None:
        print(f"--- {label}: MISSING ---")
        return
    print(f"--- {label} ---")
    print("top-level keys:", list(doc.keys()))
    if "profile" in doc:
        print("  profile =", doc["profile"])
    if "profiles" in doc:
        print("  profiles:", list(doc["profiles"].keys()))
    if "model_providers" in doc:
        for name, prov in doc["model_providers"].items():
            print(f"  model_providers.{name}.base_url =", prov.get("base_url"))
    if "mcp_servers" in doc:
        print("  mcp_servers:", list(doc["mcp_servers"].keys()))

_summary(cfg.user_doc, str(USER_CONFIG))
_summary(cfg.project_doc, str(PROJECT_CONFIG))

if cfg.catalog is not None:
    print()
    print("--- model catalog ---")
    for m in cfg.catalog.get("models", []):
        print(f"  {m['slug']:<40s}  ctx={m.get('context_window')}")


--- ~/.codex/config.toml ---
top-level keys: ['profile', 'profiles', 'model_providers']
  profile = lmstudio-qwen3-coder
  profiles: ['lmstudio-qwen3-coder', 'lmstudio-qwen35', 'lmstudio-gemma4']
  model_providers.local_lmstudio.base_url = http://127.0.0.1:18234/v1

--- /workspaces/agent-client-kernel/.codex/config.toml ---
top-level keys: ['profile', 'mcp_servers', 'profiles', 'model_providers', 'projects']
  profile = lmstudio-qwen3-coder
  profiles: ['lmstudio-qwen3-coder', 'lmstudio-qwen35', 'lmstudio-gemma4']
  model_providers.local_lmstudio.base_url = http://127.0.0.1:18234/v1
  mcp_servers: ['jupyter', 'paper-search']

--- model catalog ---
  openai/gpt-oss-120b                       ctx=131072
  qwen3.5-27b                               ctx=32768
  qwen/qwen3-coder-next                     ctx=262144
  google/gemma-4-26b-a4b                    ctx=32768


## 3. Diagnose

We build a list of `(severity, message, fix)` entries. `severity` is one of
`error` (must fix for current env to work), `warn` (likely wrong but not fatal),
or `info`. The `fix` is a callable applied later if the user opts in.

In [ ]:
diag = diagnose(env_facts, cfg)
if not diag.issues:
    print("All checks passed. Current Codex config matches the live environment.")
else:
    for sev, msg, key in diag.issues:
        tag = {"error": "[ERROR]", "warn": "[WARN] ", "info": "[INFO] "}[sev]
        suffix = f"  (fix: {key})" if key else ""
        print(f"{tag} {msg}{suffix}")


[WARN]  LM Studio has ['google/gemma-4-31b'] loaded but no profile in ~/.codex/config.toml references those models. Add a profile for it, or load one of: ['google/gemma-4-26b-a4b', 'qwen/qwen3-coder-next', 'qwen3.5-27b'].
[WARN]  project .codex/config.toml has `profile` — Codex 0.130+ ignores it at project scope. Safe to leave, but it lives in user config too.
[WARN]  project .codex/config.toml has `profiles` — Codex 0.130+ ignores it at project scope. Safe to leave, but it lives in user config too.
[WARN]  project .codex/config.toml has `model_providers` — Codex 0.130+ ignores it at project scope. Safe to leave, but it lives in user config too.
[INFO]  Preferred model 'qwen/qwen3-coder-next' not loaded. Currently loaded: ['google/gemma-4-31b']


## 4. Apply fixes

Set `APPLY = True` and re-run this cell to write the corrections back. The user
config is patched in place via `tomlkit`, so comments and any unrelated keys are
preserved. A timestamped backup is written before each change.

Only the issues that the previous cell tagged with a `fix:` key are applied —
benign extra settings are left alone.

In [ ]:
APPLY = False  # flip to True and re-run to write changes

report = apply_fixes(env_facts, cfg, diag, apply=APPLY)

planned = report["planned"]
if not planned:
    print("No fixable issues. Nothing to apply.")
else:
    print("Planned fixes:")
    for sev, msg, key in planned:
        print(f"  - [{sev}] {key}: {msg}")

if APPLY and report["written"]:
    print()
    print("Wrote:")
    for p in report["written"]:
        print(" ", p)
    print()
    print(f"--- new {USER_CONFIG} ---")
    print(USER_CONFIG.read_text())
elif planned and not APPLY:
    print()
    print("Set APPLY = True and re-run this cell to write the changes.")


No fixable issues. Nothing to apply.


## 5. Verify

Re-run cells 1–3 to confirm the diagnostic is clean, then optionally smoke-test
the active profile with a tiny `codex exec` call. This requires the `codex` CLI
on PATH and the LM Studio model loaded.

In [6]:
import subprocess

codex = env_facts.get("codex_bin")
if not codex:
    print("codex CLI not on PATH — skipping smoke test.")
else:
    try:
        out = subprocess.run(
            [codex, "--version"],
            capture_output=True, text=True, timeout=10,
        )
        print("codex --version:", out.stdout.strip() or out.stderr.strip())
    except (subprocess.TimeoutExpired, OSError) as e:
        print("codex --version failed:", e)

codex --version: codex-cli 0.130.0
